# Cross-Dataset Evaluation: PGVIB Variants

Evaluates all PGVIB checkpoint variants on four held-out OOD datasets.

**Variants evaluated:**
- Monolithic VIB baseline (3 seeds, GLADAModel)
- Full per-group VIB (3 seeds, PerGroupVIBModel, s42 from phase_a_leaveout)
- g0-only VIB (3 seeds, PerGroupVIBModel(vib_disabled_groups=(1,2)))
- drop_vib_g0 (3 seeds, PerGroupVIBModel(vib_disabled_groups=(0,)))
- Cross-backbone: WavLM-Large g0-only (phase_pgvib_g0only, seed 42)
- Cross-backbone: xlsr-300m g0-only (phase_pgvib_replicate, seed 42)

**Datasets (held-out, never used in training):**
- ASVspoof 2021 LA — telephone/VoIP codec channel conditions
- ASVspoof 2021 DF — codec-post-processed deepfakes (social media conditions)
- WaveFake — neural vocoder diversity (6 vocoders, LJSpeech bonafide)
- CodecFake+ — LM-based neural codec TTS attacks (Encodec, SoundStream, ...)

**Output:** CSV at `MyDrive/Jakob/Experiments/cross_dataset_eval_pgvib_results.csv`
with columns: variant, model_class, backbone, seed, dataset, eer, ci_low, ci_high,
auc, n_samples, checkpoint_path.

Self-contained: Run All from scratch. Missing checkpoints are skipped with a warning.
Results are written row-by-row so partial results survive session disconnects.


## 1. Environment Setup


In [ ]:
!pip install -q transformers soundfile librosa scikit-learn tqdm safetensors

from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory / 1e9:.1f} GB)')
else:
    raise RuntimeError('No GPU detected')


## 2. Copy GLADA Package to Local Runtime


In [ ]:
import shutil, os, sys

CANDIDATES = [
    "/content/drive/Othercomputers/Jakob's MacBook Air/Thesis/298-deliverables/glada-code/glada",
    '/content/drive/MyDrive/Jakob/glada_pkg/glada',
    '/content/drive/MyDrive/Jakob/glada_pkg',
    '/content/drive/MyDrive/Jakob/glada',
]
DRIVE_PKG = next((c for c in CANDIDATES if os.path.exists(os.path.join(c, '__init__.py'))), None)
if DRIVE_PKG is None:
    raise FileNotFoundError(
        'No glada/__init__.py found in candidates. Run:\n'
        "  !find /content/drive -name '__init__.py' -path '*/glada/*' 2>/dev/null | head\n"
        'and add the parent directory to CANDIDATES above.'
    )

LOCAL_PKG = '/content/glada'
if os.path.exists(LOCAL_PKG): shutil.rmtree(LOCAL_PKG)
shutil.copytree(DRIVE_PKG, LOCAL_PKG)

for mod in list(sys.modules):
    if mod == 'glada' or mod.startswith('glada.'): del sys.modules[mod]
sys.path = [p for p in sys.path if 'glada' not in p]
if '/content' not in sys.path: sys.path.insert(0, '/content')
print(f'Package source: {DRIVE_PKG}\nCopied to:      {LOCAL_PKG}')


## 3. Datasets (ASVspoof 2019 LA + ITW)


In [ ]:
import tarfile

def extract_dataset(tar_path, extract_to, name):
    if os.path.exists(extract_to) and len(os.listdir(extract_to)) > 0:
        print(f'{name}: already extracted to {extract_to}'); return
    os.makedirs(extract_to, exist_ok=True)
    print(f'{name}: extracting {tar_path} -> {extract_to} ...')
    with tarfile.open(tar_path) as t: t.extractall(extract_to)
    print(f'{name}: done.')

extract_dataset('/content/drive/MyDrive/Jakob/Datasets/LA.tar', '/content/data/asvspoof', 'ASVspoof 2019 LA')
extract_dataset('/content/drive/MyDrive/Jakob/Datasets/release_in_the_wild.tar', '/content/data/itw', 'In-the-Wild')

ASV_PROTO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_AUDIO      = '/content/data/asvspoof/LA/ASVspoof2019_LA_train/flac'
ASV_EVAL_PROTO = '/content/data/asvspoof/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt'
ASV_EVAL_AUDIO = '/content/data/asvspoof/LA/ASVspoof2019_LA_eval/flac'
ITW_META       = '/content/data/itw/release_in_the_wild/meta.csv'
ITW_AUDIO      = '/content/data/itw/release_in_the_wild'
for p in [ASV_PROTO, ASV_AUDIO, ASV_EVAL_PROTO, ITW_META]:
    assert os.path.exists(p), f'Missing: {p}'
print('Datasets verified.')


## 4. OOD Evaluation Dataset Extraction (all four held-out sets)

All four held-out evaluation datasets live on Drive (CodecFake+ on a Shared
Drive). Reading large `.flac`/`.wav` trees directly off Drive is too slow for
evaluation, so we copy & extract the archives to local runtime under
`/content/data/`. A sentinel file marks each archive as already extracted, so
re-running this cell after a session restart is cheap.

Layout produced by extraction:
```
/content/data/asvspoof2021_la/
  ASVspoof2021_LA_eval/flac/         (audio, from ASVspoof2021_LA_eval.tar.gz)
  keys/LA/CM/trial_metadata.txt      (keys,  from LA-keys-full.tar.gz)

/content/data/asvspoof2021_df/
  ASVspoof2021_DF_eval/flac/         (audio, from ASVspoof2021_DF_eval_full.tar.gz)
  keys/DF/CM/trial_metadata.txt      (keys,  from DF-keys-full.tar)

/content/data/wavefake/
  ljspeech_*/                        (spoof, from wavefake.zip)
  LJSpeech-1.1/wavs/                 (bonafide, from LJSpeech-1.1.tar.bz2)

/content/data/codecfake/
  fake/  CodecFake_plus_CoSG/...     (spoof, from CodecFake_plus_CoSG.tar.xz)
  real/  LibriSpeech/test-clean/...  (bonafide, from test-clean.tar.gz)
```

Why the `real/` wrapper for CodecFake+ bonafide: `CodecFakeDataset` uses a
path-segment heuristic — any ancestor directory whose name matches
`{real, bonafide, genuine, original, clean}` is bonafide. LibriSpeech's
`test-clean` directory does NOT match (the hyphen prevents a literal `clean`
match), so we extract under an explicit `real/` parent that does.

**Never used during GLADA training or adaptation** — these are purely held-out
test sets for measuring cross-domain generalization (Muller et al., Interspeech 2022).


In [ ]:
import os, glob, shutil, tarfile, zipfile

# Sentinel-based archive extractor.  Each archive deposits a hidden marker
# file at <extract_to>/.extracted_<basename> on success, so we can extract
# multiple archives into the same directory (audio + keys, spoof + bonafide)
# without confusing the "already extracted" check.  Auto-detects compression
# from the file extension (.tar/.tar.gz/.tar.bz2/.tar.xz/.zip).
def extract_archive(archive_path, extract_to, name):
    if not os.path.exists(archive_path):
        print(f'  [WARN] {name}: archive not found at {archive_path} — skipping')
        return False
    sentinel = os.path.join(extract_to, f'.extracted_{os.path.basename(archive_path)}')
    if os.path.exists(sentinel):
        print(f'  [SKIP] {name}: already extracted ({sentinel})')
        return True
    os.makedirs(extract_to, exist_ok=True)
    print(f'  {name}: extracting {archive_path} -> {extract_to} ...')
    if archive_path.endswith('.zip'):
        with zipfile.ZipFile(archive_path) as z:
            z.extractall(extract_to)
    else:
        # tarfile.open auto-detects .tar / .tar.gz / .tar.bz2 / .tar.xz.
        with tarfile.open(archive_path) as t:
            t.extractall(extract_to)
    open(sentinel, 'w').close()
    print(f'  {name}: done.')
    return True


def extract_split_archive(
    parts_glob, extract_to, name,
    scratch_dir='/content/data/_split_scratch',
):
    """Extract a tar.gz that was split into _part00.tar.gz / _part01.tar.gz /...
    chunks (e.g. via `split -b SIZE`).

    Use this when the canonical "_full" archive on Drive is truncated or
    missing.  The parts on Drive are NEVER modified — we copy each part to
    runtime scratch, concatenate them into one tarball, delete the per-part
    runtime copies (to free ~half the scratch usage), extract, and finally
    delete the concatenated archive.

    `cat part00 part01 ... > full.tar.gz` reproduces the original byte
    stream exactly when the parts came from `split -b ...` on the original
    .tar.gz, so tarfile then opens the reassembled archive normally.

    Disk profile (DF eval, ~80 GB across 4 parts, ~80 GB extracted FLAC):
      copy parts          ->  ~80 GB scratch
      concat              -> ~160 GB scratch (parts + concat)
      delete parts        ->  ~80 GB scratch (just concat)
      extract             -> ~80 GB scratch + ~80 GB extracted
      delete concat       -> ~80 GB extracted
    Comfortably fits Colab's 200 GB disk because FLAC barely compresses.
    """
    parts = sorted(glob.glob(parts_glob))
    if not parts:
        print(f'  [WARN] {name}: no parts matched {parts_glob} — skipping')
        return False
    sentinel = os.path.join(
        extract_to, f'.extracted_split_{os.path.basename(parts[0])}',
    )
    if os.path.exists(sentinel):
        print(f'  [SKIP] {name}: already extracted ({sentinel})')
        return True
    os.makedirs(extract_to, exist_ok=True)
    os.makedirs(scratch_dir, exist_ok=True)
    concat_path = os.path.join(
        scratch_dir,
        os.path.basename(parts[0]).replace('_part00', '_concat'),
    )

    # 1. Copy each part from Drive -> runtime scratch (skip if already there).
    local_parts = []
    for src in parts:
        dst = os.path.join(scratch_dir, os.path.basename(src))
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
            print(f'  {name}: copying {os.path.basename(src)} '
                  f'({os.path.getsize(src) / 1e9:.1f} GB) -> {scratch_dir}')
            shutil.copy2(src, dst)
        local_parts.append(dst)

    # 2. Concatenate runtime parts into one tarball.
    print(f'  {name}: concatenating {len(local_parts)} parts -> {concat_path}')
    with open(concat_path, 'wb') as out:
        for p in local_parts:
            with open(p, 'rb') as inp:
                shutil.copyfileobj(inp, out, length=8 * 1024 * 1024)  # 8 MiB

    # 3. Delete the per-part runtime copies (Drive originals untouched).
    for p in local_parts:
        os.remove(p)
    print(f'  {name}: deleted {len(local_parts)} runtime parts '
          f'(Drive originals preserved)')

    # 4. Extract from the concatenated tarball.
    print(f'  {name}: extracting {concat_path} -> {extract_to}')
    with tarfile.open(concat_path) as t:
        t.extractall(extract_to)

    # 5. Delete the concatenated archive — frees the rest of scratch.
    os.remove(concat_path)
    open(sentinel, 'w').close()
    print(f'  {name}: done (concat archive deleted; only extracted tree remains).')
    return True


# ---- ASVspoof 2021 LA: extract audio + keys into one root ----
ASV21_LA_ROOT = '/content/data/asvspoof2021_la'
extract_archive(
    '/content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_LA/ASVspoof2021_LA_eval.tar.gz',
    ASV21_LA_ROOT,
    'ASVspoof 2021 LA audio',
)
extract_archive(
    '/content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_LA/LA-keys-full.tar.gz',
    ASV21_LA_ROOT,
    'ASVspoof 2021 LA keys',
)
ASV21_LA_KEYS  = f'{ASV21_LA_ROOT}/keys/LA/CM/trial_metadata.txt'
ASV21_LA_AUDIO = f'{ASV21_LA_ROOT}/ASVspoof2021_LA_eval/flac'

# ---- ASVspoof 2021 DF: extract audio (from split parts) + keys into one root ----
# `_full.tar.gz` on Drive is truncated (upload failure), so we reassemble
# the audio from the four `_part0N.tar.gz` chunks via extract_split_archive.
# Keys ship in a single .tar — extract normally.
ASV21_DF_ROOT = '/content/data/asvspoof2021_df'
extract_split_archive(
    '/content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF/ASVspoof2021_DF_eval_part*.tar.gz',
    ASV21_DF_ROOT,
    'ASVspoof 2021 DF audio',
)
extract_archive(
    '/content/drive/MyDrive/Jakob/Datasets/ASVspoof2021_DF/DF-keys-full.tar',
    ASV21_DF_ROOT,
    'ASVspoof 2021 DF keys',
)
ASV21_DF_KEYS  = f'{ASV21_DF_ROOT}/keys/DF/CM/trial_metadata.txt'
ASV21_DF_AUDIO = f'{ASV21_DF_ROOT}/ASVspoof2021_DF_eval/flac'

# ---- WaveFake (spoof) + LJSpeech-1.1 (bonafide) into one root ----
# WaveFakeDataset detects bonafide by scanning for 'ljspeech-1.1' in any
# subdir path (case-insensitive), so spoof + bonafide can coexist here.
WAVEFAKE_ROOT = '/content/data/wavefake'
extract_archive(
    '/content/drive/MyDrive/Jakob/Datasets/wavefake.zip',
    WAVEFAKE_ROOT,
    'WaveFake spoof (vocoders)',
)
extract_archive(
    '/content/drive/MyDrive/Jakob/Datasets/LJSpeech-1.1.tar.bz2',
    WAVEFAKE_ROOT,
    'LJSpeech-1.1 (WaveFake bonafide)',
)

# ---- CodecFake+ (spoof under fake/, LibriSpeech test-clean under real/) ----
# CodecFakeDataset uses an ancestor-name heuristic; see markdown above.
CODECFAKE_ROOT = '/content/data/codecfake'
extract_archive(
    '/content/drive/Shareddrives/Jakob-298-datasets/CodecFakePlus/CodecFake_plus_CoSG.tar.xz',
    f'{CODECFAKE_ROOT}/fake',
    'CodecFake+ CoSG (spoof)',
)
extract_archive(
    '/content/drive/Shareddrives/Jakob-298-datasets/CodecFakePlus/test-clean.tar.gz',
    f'{CODECFAKE_ROOT}/real',
    'LibriSpeech test-clean (CodecFake bonafide)',
)

# Verify paths exist — warn but do not hard-abort; missing datasets are
# skipped gracefully by the DataLoader-build cell below.
DATASET_PATHS = {
    '2021_LA':   (ASV21_LA_KEYS, ASV21_LA_AUDIO),
    '2021_DF':   (ASV21_DF_KEYS, ASV21_DF_AUDIO),
    'WaveFake':  (WAVEFAKE_ROOT,),
    'CodecFake': (CODECFAKE_ROOT,),
}
print()
for ds_name, paths in DATASET_PATHS.items():
    for p in paths:
        if os.path.exists(p):
            print(f'  OK   {ds_name}: {p}')
        else:
            print(f'  WARN {ds_name}: NOT FOUND — {p}')


## 3. Imports


In [ ]:
import gc, json, os, sys, time, warnings
import csv
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# GLADA package (copied in Section 1)
from glada import GLADAConfig, BACKBONE_REGISTRY, set_seed
from glada.models import GLADAModel
from glada.models.vib import VIBBottleneck
from glada.models.grl import GradientReversal
from glada.models.discriminators import GroupedLayerDomainDiscriminators
from glada.evaluation.metrics import compute_eer_with_ci

# OOD dataset classes — reuse existing loaders, do NOT reinvent.
from glada.datasets.asvspoof2021 import ASVspoof2021Dataset
from glada.datasets.wavefake import WaveFakeDataset
from glada.datasets.codecfake import CodecFakeDataset

print('Imports OK')


## 4. PerGroupVIBModel Definition

Identical to the inline definition in `_phase_common.model_class_cells()`. Duplicated here
so this notebook is self-contained (Run All from scratch).

The `vib_disabled_groups` and `vib_betas` flags must match the flags used during training;
they are recorded in `VARIANTS` below and cross-checked against the checkpoint's state dict
shape before loading.


In [ ]:
class PerGroupVIBModel(GLADAModel):
    """GLADAModel with one VIBBottleneck per group, applied BEFORE the
    per-group discriminators. See _phase_common.model_class_cells() for full
    docstring and literature references (VIB: Alemi et al. 2017; GLADA: MLADA).

    This definition is duplicated here to keep the eval notebook self-contained.
    It must stay in sync with the training notebook definition in _phase_common.py.
    """

    def __init__(self, config, vib_disabled_groups=(), vib_betas=None):
        super().__init__(config)
        K = config.num_groups
        D = config.hidden_dim
        bd = config.bottleneck_dim
        self.group_vibs = nn.ModuleList([
            VIBBottleneck(2 * D, bd) for _ in range(K)
        ])
        self.group_compressed_proj = nn.Linear(K * bd, bd)
        if self.glada_disc is not None:
            self.glada_disc = GroupedLayerDomainDiscriminators(
                num_groups=K,
                input_dim=bd,
                hidden_dim=config.glada_disc_hidden,
            )
        for k in vib_disabled_groups:
            assert 0 <= k < K, f'vib_disabled_groups index {k} not in [0, {K})'
        self._vib_disabled = set(vib_disabled_groups)
        if vib_betas is not None:
            assert len(vib_betas) == K
            self.register_buffer('_vib_betas',
                torch.tensor(vib_betas, dtype=torch.float32))
        else:
            self._vib_betas = None

    def forward(self, input_values, attention_mask=None):
        backbone_frozen = not any(p.requires_grad for p in self.backbone.parameters())
        if backbone_frozen:
            with torch.no_grad():
                outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        else:
            outputs = self.backbone(input_values, attention_mask=attention_mask, output_hidden_states=True)
        hidden_states = list(outputs.hidden_states)

        if attention_mask is not None:
            input_lengths = attention_mask.sum(dim=1)
            output_lengths = ((input_lengths - 1) // 320) + 1
            T = hidden_states[0].shape[1]
            frame_mask = (torch.arange(T, device=attention_mask.device).unsqueeze(0) < output_lengths.unsqueeze(1))
        else:
            frame_mask = None

        hidden_states = self.layer_norms(hidden_states)
        _, cka_loss, group_outputs = self.pooling(hidden_states, frame_mask=frame_mask)
        if not self.config.use_cka_loss:
            cka_loss = torch.tensor(0.0, device=group_outputs[0].device)

        z_groups = []
        kl_per_group = []
        for k, g_k in enumerate(group_outputs):
            if k in self._vib_disabled:
                z_k, _ = self.group_vibs[k](g_k, training=False)
                kl_k = torch.zeros((), device=g_k.device)
            else:
                z_k, kl_k = self.group_vibs[k](g_k, training=self.training)
            z_groups.append(z_k)
            kl_per_group.append(kl_k)

        if self._vib_betas is not None:
            kl_total = sum(self._vib_betas[k] * kl_per_group[k] for k in range(len(group_outputs)))
        else:
            kl_total = sum(kl_per_group) / max(len(group_outputs), 1)

        z_concat = torch.cat(z_groups, dim=-1)
        z = self.group_compressed_proj(z_concat)

        spoof_logits = self.task_classifier(z)
        result = {
            'spoof_logits': spoof_logits,
            'kl_loss': kl_total,
            'cka_loss': cka_loss,
            'features': z,
        }

        if self.glada_disc is not None:
            result['glada_domain_logits'] = self.glada_disc(z_groups, self.domain_grl_alpha)
            result['group_outputs'] = z_groups

        if self.domain_disc is not None:
            z_domain = GradientReversal.apply(z, self.domain_grl_alpha)
            result['domain_logits'] = self.domain_disc(z_domain)

        if self.speaker_adversary is not None:
            z_speaker = GradientReversal.apply(z, self.speaker_grl_alpha)
            result['speaker_logits'] = self.speaker_adversary(z_speaker)

        return result

print('PerGroupVIBModel defined.')


## 5. Variant Configuration

Each entry specifies:
- `phase_dir`: Drive subdirectory under `MyDrive/Jakob/Experiments/`
  (also used to locate the results JSON for best-epoch lookup)
- `results_json`: filename of the results JSON saved by the training notebook
- `model_class`: `'GLADAModel'` or `'PerGroupVIBModel'`
- `model_kwargs`: forwarded to `ModelCls(config, **model_kwargs)`
- `backbone`: SSL backbone name (default `wavlm-base-plus`)
- `seed`: training seed (used to reconstruct `GLADAConfig`)

**pergroup_vib_s42 and drop_vib_g0_s42** are from `phase_a_leaveout`, NOT
`phase_pgvib_replicate`. All other per-group / drop-g0 variants at seeds 123/456
are from `phase_pgvib_replicate`.


In [ ]:
DRIVE_EXP_BASE = '/content/drive/MyDrive/Jakob/Experiments'
OUTPUT_CSV     = os.path.join(DRIVE_EXP_BASE, 'cross_dataset_eval_pgvib_results.csv')

# -------------------------------------------------------------------------
# Variant table.  checkpoint_dir convention (set by GLADAConfig / base_trainer):
#   <phase_dir>/<variant_name>/<run_id>/checkpoint_epoch_<best_epoch>.pt
# The run_id is a timestamp string (YYYYmmdd_HHMMSS).  We glob for it.
# -------------------------------------------------------------------------
VARIANTS = {
    # ---- Monolithic VIB baseline (from phase_pgvib_monolithic) ----
    'monolithic_vib_s42': {
        'phase_dir':    'phase_pgvib_monolithic',
        'results_json': 'phase_pgvib_monolithic_results.json',
        'model_class':  'GLADAModel',
        'model_kwargs': {},
        'backbone':     'wavlm-base-plus',
        'seed':         42,
    },
    'monolithic_vib_s123': {
        'phase_dir':    'phase_pgvib_monolithic',
        'results_json': 'phase_pgvib_monolithic_results.json',
        'model_class':  'GLADAModel',
        'model_kwargs': {},
        'backbone':     'wavlm-base-plus',
        'seed':         123,
    },
    'monolithic_vib_s456': {
        'phase_dir':    'phase_pgvib_monolithic',
        'results_json': 'phase_pgvib_monolithic_results.json',
        'model_class':  'GLADAModel',
        'model_kwargs': {},
        'backbone':     'wavlm-base-plus',
        'seed':         456,
    },

    # ---- Full per-group VIB (s42 from phase_a_leaveout; s123/s456 from phase_pgvib_replicate) ----
    'pergroup_vib_s42': {
        'phase_dir':    'phase_a_leaveout',
        'variant_dir':  'sanity_no_disable',
        'results_json': 'phase_a_leaveout_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {},
        'backbone':     'wavlm-base-plus',
        'seed':         42,
    },
    'pergroup_vib_s123': {
        'phase_dir':    'phase_pgvib_replicate',
        'results_json': 'phase_pgvib_replicate_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {},
        'backbone':     'wavlm-base-plus',
        'seed':         123,
    },
    'pergroup_vib_s456': {
        'phase_dir':    'phase_pgvib_replicate',
        'results_json': 'phase_pgvib_replicate_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {},
        'backbone':     'wavlm-base-plus',
        'seed':         456,
    },

    # ---- g0-only VIB (phase_pgvib_g0only) ----
    'g0_only_vib_s42': {
        'phase_dir':    'phase_pgvib_g0only',
        'results_json': 'phase_pgvib_g0only_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (1, 2)},
        'backbone':     'wavlm-base-plus',
        'seed':         42,
    },
    'g0_only_vib_s123': {
        'phase_dir':    'phase_pgvib_g0only',
        'results_json': 'phase_pgvib_g0only_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (1, 2)},
        'backbone':     'wavlm-base-plus',
        'seed':         123,
    },
    'g0_only_vib_s456': {
        'phase_dir':    'phase_pgvib_g0only',
        'results_json': 'phase_pgvib_g0only_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (1, 2)},
        'backbone':     'wavlm-base-plus',
        'seed':         456,
    },

    # ---- drop_vib_g0 (s42 from phase_a_leaveout; s123/s456 from phase_pgvib_replicate) ----
    'drop_vib_g0_s42': {
        'phase_dir':    'phase_a_leaveout',
        'variant_dir':  'drop_vib_g0',
        'results_json': 'phase_a_leaveout_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (0,)},
        'backbone':     'wavlm-base-plus',
        'seed':         42,
    },
    'drop_vib_g0_s123': {
        'phase_dir':    'phase_pgvib_replicate',
        'results_json': 'phase_pgvib_replicate_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (0,)},
        'backbone':     'wavlm-base-plus',
        'seed':         123,
    },
    'drop_vib_g0_s456': {
        'phase_dir':    'phase_pgvib_replicate',
        'results_json': 'phase_pgvib_replicate_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (0,)},
        'backbone':     'wavlm-base-plus',
        'seed':         456,
    },

    # ---- Cross-backbone: WavLM-Large g0-only (phase_pgvib_g0only) ----
    # Large has 4 groups → vib_disabled_groups=(1, 2, 3)
    'g0_only_large_s42': {
        'phase_dir':    'phase_pgvib_g0only',
        'results_json': 'phase_pgvib_g0only_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (1, 2, 3)},
        'backbone':     'wavlm-large',
        'seed':         42,
    },

    # ---- Cross-backbone: xlsr-300m g0-only (phase_pgvib_replicate) ----
    # xlsr-300m has 4 groups → vib_disabled_groups=(1, 2, 3)
    'g0_only_xlsr_s42': {
        'phase_dir':    'phase_pgvib_replicate',
        'results_json': 'phase_pgvib_replicate_results.json',
        'model_class':  'PerGroupVIBModel',
        'model_kwargs': {'vib_disabled_groups': (1, 2, 3)},
        'backbone':     'xlsr-300m',
        'seed':         42,
    },
}

print(f'Total variants: {len(VARIANTS)}')
for name, spec in VARIANTS.items():
    print(f'  {name:<28s} cls={spec["model_class"]:<18s} backbone={spec["backbone"]:<20s} seed={spec["seed"]}')


## 6. Checkpoint Resolution Helpers

Two helpers:
1. `find_best_checkpoint(phase_dir, results_json, variant_name)` — reads the phase
   results JSON, extracts `best_eer`'s epoch, then globs for a matching checkpoint file.
2. `load_model(variant_name, spec, device)` — constructs a `GLADAConfig`, instantiates
   the right model class, loads state dict, and returns the model in eval mode.

Checkpoint naming convention (from `base_trainer.py`):
  `<DRIVE_EXP_BASE>/<phase_dir>/<variant_name>/<run_id>/checkpoint_epoch_<N>.pt`
The `run_id` is a timestamp; we glob `<variant_name>/*/checkpoint_epoch_<N>.pt`.
If multiple run_ids exist (re-runs), we take the most recent one.


In [ ]:
MODEL_CLASSES = {'GLADAModel': GLADAModel, 'PerGroupVIBModel': PerGroupVIBModel}


def find_best_checkpoint(phase_dir, variant_dir):
    """Locate the best checkpoint for a variant.

    Prefers `best_model.pt` (canonical best-EER snapshot, always saved with
    the embedded config dict). Falls back to the newest `ckpt_ep*.pt`
    periodic checkpoint if best_model.pt is missing. We do NOT use
    best_model.safetensors here because it has no embedded GLADAConfig
    and we need it to rebuild the model with the trained structural
    hyperparameters (num_speakers, num_groups, hidden_dim, ...).
    """
    import glob as _glob
    phase_path = os.path.join(DRIVE_EXP_BASE, phase_dir)
    var_path   = os.path.join(phase_path, variant_dir)

    matches = sorted(
        _glob.glob(os.path.join(var_path, '*', 'best_model.pt')),
        key=os.path.getmtime,
    )
    if matches:
        return matches[-1]

    matches = sorted(
        _glob.glob(os.path.join(var_path, '*', 'ckpt_ep*.pt')),
        key=os.path.getmtime,
    )
    if matches:
        print(f'  [INFO] no best_model.pt under {var_path}; using newest periodic '
              f'ckpt: {os.path.basename(matches[-1])}')
        return matches[-1]

    print(f'  [WARN] no checkpoint found under {var_path}')
    return None


def _state_dict_from(ckpt):
    """Unwrap whichever key the trainer used."""
    if isinstance(ckpt, dict):
        if 'model' in ckpt and isinstance(ckpt['model'], dict):
            return ckpt['model']
        if 'model_state_dict' in ckpt:
            return ckpt['model_state_dict']
    return ckpt


def load_model(variant_name, spec, device):
    """Instantiate, load state dict, return model in eval mode.

    The trainer wraps `asdict(self.config)` under the key `'config'`
    inside each .pt checkpoint (base_trainer.py:_save_checkpoint). We
    rebuild GLADAConfig from that dict so structural fields trained into
    the checkpoint (num_speakers, num_groups, hidden_dim, bottleneck_dim,
    backbone) match the saved tensors. Without this step, defaults like
    num_speakers=107 collide with the trained num_speakers=20 and
    load_state_dict fails on speaker_adversary.6.

    `variant_dir` (in spec) overrides `variant_name` for path resolution
    when the on-disk directory was named differently from the dict key.

    Returns:
        (model, ckpt_path, config) or (None, None, None) if loading failed.
    """
    variant_dir = spec.get('variant_dir', variant_name)
    ckpt_path = find_best_checkpoint(spec['phase_dir'], variant_dir)
    if ckpt_path is None:
        return None, None, None

    try:
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    except Exception as e:
        print(f'  [ERROR] torch.load failed for {ckpt_path}: {e}')
        return None, None, None

    cfg_dict = ckpt.get('config', {}) if isinstance(ckpt, dict) else {}
    # Restrict to fields GLADAConfig actually accepts (forward-compat with
    # added/removed fields between training and analysis).
    valid_fields = {k: v for k, v in cfg_dict.items()
                    if k in GLADAConfig.__dataclass_fields__}
    # Override eval-time settings without disturbing structural fields.
    valid_fields['experiment_name'] = variant_name
    valid_fields['batch_size']      = 64
    valid_fields['num_workers']     = 0
    valid_fields['seed']            = spec['seed']
    valid_fields['backbone']        = spec['backbone']
    cfg = GLADAConfig(**valid_fields)

    ModelCls = MODEL_CLASSES[spec['model_class']]
    model_kwargs = spec.get('model_kwargs', {})

    try:
        model = ModelCls(cfg, **model_kwargs).to(device)
        state_dict = _state_dict_from(ckpt)
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        if missing:
            print(f'  [INFO] {variant_name}: {len(missing)} missing keys (e.g. {list(missing)[:3]})')
        if unexpected:
            print(f'  [INFO] {variant_name}: {len(unexpected)} unexpected keys')
        model.eval()
        n_params = sum(p.numel() for p in model.parameters())
        print(f'  Loaded {variant_name}: {n_params:,} params  '
              f'(num_speakers={cfg.num_speakers}, num_groups={cfg.num_groups})  '
              f'from {ckpt_path}')
        return model, ckpt_path, cfg
    except Exception as e:
        import traceback; traceback.print_exc()
        print(f'  [ERROR] failed to load {variant_name}: {e}')
        return None, None, None


## 7. Build OOD DataLoaders

Constructs a DataLoader for each available OOD dataset.
Datasets whose paths do not exist on Drive are skipped with a warning — the
eval loop below handles missing loaders gracefully.

batch_size=64 for inference (larger than training batch_size=24-64) since we
have no optimizer state and can afford more GPU memory per forward pass.
num_workers=0 per project rule (Colab multiprocessing issues).


In [ ]:
EVAL_BATCH_SIZE = 64
MAX_LEN = 64000  # 4 seconds at 16 kHz — matches training

eval_loaders = {}  # dataset_name -> DataLoader (or None if unavailable)

# ---- ASVspoof 2021 LA ----
try:
    if os.path.exists(ASV21_LA_KEYS) and os.path.exists(ASV21_LA_AUDIO):
        ds_2021_la = ASVspoof2021Dataset(
            keys_path=ASV21_LA_KEYS,
            audio_dir=ASV21_LA_AUDIO,
            max_len=MAX_LEN,
        )
        eval_loaders['2021_LA'] = DataLoader(
            ds_2021_la, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0,
        )
    else:
        print('[WARN] ASVspoof 2021 LA not found — skipping.')
except Exception as e:
    print(f'[WARN] ASVspoof 2021 LA load error: {e}')

# ---- ASVspoof 2021 DF ----
try:
    if os.path.exists(ASV21_DF_KEYS) and os.path.exists(ASV21_DF_AUDIO):
        ds_2021_df = ASVspoof2021Dataset(
            keys_path=ASV21_DF_KEYS,
            audio_dir=ASV21_DF_AUDIO,
            max_len=MAX_LEN,
        )
        eval_loaders['2021_DF'] = DataLoader(
            ds_2021_df, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0,
        )
    else:
        print('[WARN] ASVspoof 2021 DF not found — skipping.')
except Exception as e:
    print(f'[WARN] ASVspoof 2021 DF load error: {e}')

# ---- WaveFake ----
try:
    if os.path.exists(WAVEFAKE_ROOT):
        ds_wavefake = WaveFakeDataset(
            root_dir=WAVEFAKE_ROOT,
            max_len=MAX_LEN,
        )
        if len(ds_wavefake) > 0:
            eval_loaders['WaveFake'] = DataLoader(
                ds_wavefake, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0,
            )
        else:
            print('[WARN] WaveFake dataset is empty — skipping.')
    else:
        print('[WARN] WaveFake root not found — skipping.')
except Exception as e:
    print(f'[WARN] WaveFake load error: {e}')

# ---- CodecFake+ ----
try:
    if os.path.exists(CODECFAKE_ROOT):
        ds_codecfake = CodecFakeDataset(
            root_dir=CODECFAKE_ROOT,
            max_len=MAX_LEN,
        )
        if len(ds_codecfake) > 0:
            eval_loaders['CodecFake'] = DataLoader(
                ds_codecfake, batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=0,
            )
        else:
            print('[WARN] CodecFake+ dataset is empty — skipping.')
    else:
        print('[WARN] CodecFake+ root not found — skipping.')
except Exception as e:
    print(f'[WARN] CodecFake+ load error: {e}')

print(f'\nAvailable eval loaders: {list(eval_loaders.keys())}')
for name, loader in eval_loaders.items():
    print(f'  {name}: {len(loader.dataset):,} samples, {len(loader)} batches')


## 8. Main Evaluation Loop

For each variant x dataset:
1. Load best checkpoint (skips gracefully if not found).
2. Run inference — `model.eval()` + `torch.no_grad()`. BF16 AMP on A100,
   float16 on T4 (the evaluator handles dtype selection).
3. Compute EER + AUC via `glada.evaluation.metrics.compute_eer_with_ci`
   (1000 bootstrap resamples for 95% CI — required for committee-ready results).
4. Append a row to the CSV and free GPU memory.

Primary metric: EER (Equal Error Rate, lower is better; 50% = chance).
Secondary metric: AUC-ROC (higher is better).
Both established in the ASVspoof challenge series (Yamagishi et al., 2021).


In [ ]:
@torch.no_grad()
def run_inference(model, loader, device, use_bf16=True):
    """Collect scores and labels from a DataLoader.

    Uses AMP (bfloat16 on A100 / float16 on T4) to match training-time
    numerical precision. BFloat16 tensors must be cast to float32 before
    numpy conversion (known PyTorch limitation; see evaluator.py comments).

    Returns:
        scores (np.ndarray): bonafide probability P(class=1|x)
        labels (np.ndarray): ground truth (0=spoof, 1=bonafide)
    """
    model.eval()
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    scores_list, labels_list = [], []
    for batch in tqdm(loader, leave=False):
        inp  = batch['input_values'].to(device, non_blocking=True)
        mask = batch['attention_mask'].to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=True, dtype=amp_dtype):
            out = model(inp, attention_mask=mask)
        # Bonafide probability: P(class=1 | x) via softmax over spoof_logits.
        probs = F.softmax(out['spoof_logits'], dim=1)[:, 1]
        # Cast to float32 before numpy conversion (BF16 limitation).
        scores_list.extend(probs.float().cpu().numpy())
        labels_list.extend(batch['task_label'].numpy())
    return np.array(scores_list), np.array(labels_list)


# ---- Setup ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
print(f'Device: {device}  BF16: {use_bf16}')

# CSV output: initialize with header.
CSV_COLUMNS = ['variant', 'model_class', 'backbone', 'seed', 'dataset',
               'eer', 'ci_low', 'ci_high', 'auc', 'n_samples', 'checkpoint_path']
if not os.path.exists(OUTPUT_CSV):
    with open(OUTPUT_CSV, 'w', newline='') as f:
        csv.writer(f).writerow(CSV_COLUMNS)
    print(f'Created CSV: {OUTPUT_CSV}')
else:
    print(f'Appending to existing CSV: {OUTPUT_CSV}')

# ---- Main loop ----
all_rows = []
loop_start = time.time()

for v_idx, (variant_name, spec) in enumerate(VARIANTS.items()):
    print(f'\n{"="*60}')
    print(f'[{v_idx+1}/{len(VARIANTS)}] {variant_name}')
    print(f'  model_class={spec["model_class"]}  backbone={spec["backbone"]}  seed={spec["seed"]}')

    # Load checkpoint — skip gracefully if not found.
    model, ckpt_path, cfg = load_model(variant_name, spec, device)
    if model is None:
        print(f'  [SKIP] {variant_name} — checkpoint not available.')
        continue

    # Evaluate on each available OOD dataset.
    for ds_name, loader in eval_loaders.items():
        t0 = time.time()
        try:
            scores, labels = run_inference(model, loader, device, use_bf16=use_bf16)
            metrics = compute_eer_with_ci(labels, scores, n_bootstrap=1000)
            elapsed = time.time() - t0
            print(
                f'  {ds_name:<12s}: EER={metrics["eer"]:.2f}%  '
                f'CI=[{metrics["ci_low"]:.2f}, {metrics["ci_high"]:.2f}]  '
                f'AUC={metrics["auc"]:.4f}  n={metrics["acc"]:.0f}%acc  '
                f'({elapsed:.0f}s)'
            )
            row = {
                'variant':         variant_name,
                'model_class':     spec['model_class'],
                'backbone':        spec['backbone'],
                'seed':            spec['seed'],
                'dataset':         ds_name,
                'eer':             round(metrics['eer'], 4),
                'ci_low':          round(metrics['ci_low'], 4),
                'ci_high':         round(metrics['ci_high'], 4),
                'auc':             round(metrics['auc'], 6),
                'n_samples':       len(labels),
                'checkpoint_path': ckpt_path,
            }
            all_rows.append(row)
            # Write immediately so partial results survive session disconnects.
            with open(OUTPUT_CSV, 'a', newline='') as f:
                w = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
                w.writerow(row)
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'  [ERROR] {variant_name} x {ds_name}: {e}')

    # Free GPU memory between checkpoints.
    del model
    torch.cuda.empty_cache(); gc.collect()

total_min = (time.time() - loop_start) / 60
print(f'\nEval complete: {len(all_rows)} rows in {total_min:.1f} min')
print(f'Results: {OUTPUT_CSV}')


## 9. Summary Table

Groups results by condition (model_class + backbone) and computes
multi-seed mean ± std for each (condition, dataset) pair.
Print order: conditions as rows, datasets as columns.


In [ ]:
import statistics

if not all_rows:
    print('No results to summarize — all variants were skipped or failed.')
else:
    # Build pandas-like table from raw rows.
    import collections
    # Group by (model_class, backbone, dataset) -> list of EERs
    groups = collections.defaultdict(list)
    for row in all_rows:
        key = (row['model_class'], row['backbone'], row['dataset'])
        groups[key].append(row['eer'])

    ds_names = list(eval_loaders.keys()) if eval_loaders else ['2021_LA', '2021_DF', 'WaveFake', 'CodecFake']

    # Header
    col_w = 14
    header = f'{"condition":<38s}' + ''.join(f'{d:>{col_w}s}' for d in ds_names)
    print(header)
    print('-' * len(header))

    # Unique conditions (ordered by first appearance)
    seen_conditions = []
    condition_of = {}  # variant -> (model_class, backbone)
    for row in all_rows:
        cond = (row['model_class'], row['backbone'])
        condition_of[row['variant']] = cond
        if cond not in seen_conditions:
            seen_conditions.append(cond)

    for cond in seen_conditions:
        label = f'{cond[0]} / {cond[1]}'
        cells = []
        for ds in ds_names:
            key = (*cond, ds)
            eers = groups.get(key, [])
            if not eers:
                cells.append('-')
            elif len(eers) == 1:
                cells.append(f'{eers[0]:.2f}')
            else:
                m = statistics.mean(eers)
                s = statistics.stdev(eers)
                cells.append(f'{m:.2f}±{s:.2f}')
        row_str = f'{label:<38s}' + ''.join(f'{c:>{col_w}s}' for c in cells)
        print(row_str)

    print()
    print('Format: mean (EER%) or mean±std across seeds. Lower is better.')
    print(f'Full results at: {OUTPUT_CSV}')
